<a href="https://colab.research.google.com/github/darlim9141/kcu5/blob/main/kmeans_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Study Note] Fashion Style Clustering & Latent Space Analysis

## 1. Introduction

### 1.1. Background & Objective
Before proceeding with **Supervised Learning** (training a classifier with labels), it is crucial to perform **Unsupervised Learning** to validate the quality of the collected dataset and analyze the intrinsic visual similarities between images.

* **Core Question:** How does a computer mathematically distinguish abstract fashion styles (e.g., 'Minimal', 'Street') without being provided with explicit labels?
* **Goal:** To extract high-dimensional feature vectors from images, project them into a lower-dimensional latent space for visualization, and verify if the data naturally groups into distinct clusters using the K-Means algorithm.

### 1.2. Role in this Project
We collected images for four distinct styles. However, we must verify if this labeling is visually valid or if it contains human bias. By excluding label information and observing how the algorithm groups the images, we can objectively assess the visual distribution of the dataset.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

# 1. Load Data (Pre-extracted Feature Vectors)
# Note: In a real scenario, these features are extracted via VGG16/ResNet50.
try:
    # Update the path according to your environment
    features = np.load('image_features.npy')
    true_labels = np.load('image_labels.npy')
    print(f"[Info] Data Loaded Successfully. Shape: {features.shape}")
except FileNotFoundError:
    print("[Warning] Data files not found. Generating dummy data for demonstration.")
    # Generating random data for demonstration purposes (400 images, 4096 features)
    features = np.random.rand(400, 4096)
    true_labels = np.random.randint(0, 4, 400)

## 2. Theoretical Background

### 2.1. Feature Extraction
* **Definition:** The process of converting raw pixel values (which are sensitive to noise and variations) into a compact numerical vector that represents the semantic information (edges, textures, patterns, shapes) of an image.
* **VGG16 Model:** A Convolutional Neural Network (CNN) developed by the Visual Geometry Group at Oxford. It is pre-trained on the ImageNet dataset (1.2 million images), providing robust capability in recognizing general visual features.
* **Transfer Learning:** A technique where knowledge learned from one task is applied to a different but related problem. In this project, we utilize the **Convolutional Base** of VGG16 while removing the final classification layers to extract a 4,096-dimensional feature vector (Embedding).

### 2.2. Dimensionality Reduction: PCA
* **Curse of Dimensionality:** As the number of dimensions (features) increases, the data becomes sparse, and distance metrics (like Euclidean distance) become less meaningful, degrading the performance of clustering algorithms.
* **PCA (Principal Component Analysis):** A linear dimensionality reduction technique that identifies a new set of orthogonal axes (Principal Components) that maximize the variance of the data, thereby preserving the most critical information while reducing dimensions.

#### **Pros & Cons of PCA**
| Aspect | Description |
| :--- | :--- |
| **Pros** | 1. **Computational Efficiency:** Reduces data size, significantly speeding up subsequent algorithms like K-Means.<br>2. **Noise Reduction:** Removing lower-variance components helps eliminate noise from the data.<br>3. **Visualization:** Enables plotting of multi-dimensional data in 2D or 3D space. |
| **Cons** | 1. **Linearity Assumption:** PCA assumes data is linearly distributed. It may fail to capture complex non-linear structures.<br>2. **Information Loss:** Reducing dimensions inevitably leads to some loss of information.<br>3. **Interpretability:** Principal components are linear combinations of original features, making them difficult to interpret intuitively. |

## 3. Clustering Algorithm: K-Means

### 3.1. Overview
**K-Means** is an iterative algorithm that partitions the dataset into $K$ distinct, non-overlapping subgroups (clusters). It operates by minimizing the variance (Euclidean distance) within each cluster.

### 3.2. Algorithm Steps
The algorithm converges through the following iterative process:

1.  **Initialization:** Randomly select $K$ data points as initial centroids.
2.  **Assignment:** Assign each data point to the nearest centroid based on Euclidean distance.
3.  **Update:** Recalculate the centroids by taking the mean of all data points assigned to each cluster.
4.  **Repeat:** Repeat steps 2 and 3 until the centroids no longer change (convergence).



[Image of K-Means algorithm steps]


### 3.3. Application
Since we assume our dataset consists of 4 distinct styles (Minimal, Casual, Classic, Street), we set the hyperparameter $K=4$. If the clusters derived by the algorithm match our ground truth labels, it validates that our dataset has distinct visual patterns.

In [ ]:
# --- Step 1: Dimensionality Reduction using PCA ---
# Reducing 4096 dimensions to 50 dimensions to avoid the Curse of Dimensionality
print("1. Applying PCA...")
pca = PCA(n_components=50, random_state=42)
features_pca = pca.fit_transform(features)

print(f"   - Original Shape: {features.shape}")
print(f"   - Reduced Shape:  {features_pca.shape}")
print(f"   - Cumulative Explained Variance: {np.sum(pca.explained_variance_ratio_):.2%}")


# --- Step 2: K-Means Clustering ---
# Grouping data into 4 clusters based on visual similarity
print("\n2. Executing K-Means Clustering (K=4)...")
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features_pca)

# Evaluation using Silhouette Score (Range: -1 to 1, higher is better)
sil_score = silhouette_score(features_pca, cluster_labels)
print(f"   - Silhouette Score: {sil_score:.4f}")

## 4. Visualization & Analysis (t-SNE)

### 4.1. t-SNE (t-Distributed Stochastic Neighbor Embedding)
While PCA is excellent for preserving global variance, **t-SNE** is a non-linear technique specifically designed for visualization. It excels at preserving the **local structure** of high-dimensional data, meaning points that are close in the high-dimensional space remain close in the 2D projection.

### 4.2. Experiment Results
We visualize the data distribution in two ways:
1.  **Ground Truth Distribution:** Coloring points based on the actual style labels.
2.  **Predicted Cluster Distribution:** Coloring points based on K-Means results.

In [ ]:
# --- Step 3: Visualization using t-SNE ---
print("\n3. Applying t-SNE for 2D Visualization...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_tsne = tsne.fit_transform(features_pca)

def plot_clusters(data, labels, title):
    plt.figure(figsize=(10, 8))
    unique_labels = np.unique(labels)

    # Define colors or colormap if needed
    for label in unique_labels:
        indices = np.where(labels == label)
        plt.scatter(data[indices, 0], data[indices, 1], label=f'Class/Cluster {label}', alpha=0.6, edgecolors='w', s=50)

    plt.title(title, fontsize=14)
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Visualize: Actual Labels vs. K-Means Predicted Clusters
plot_clusters(features_tsne, true_labels, 'Figure 1. Latent Space Distribution by Ground Truth Labels')
plot_clusters(features_tsne, cluster_labels, 'Figure 2. Latent Space Distribution by K-Means Clusters')

## 5. Conclusion and Insights

Based on the t-SNE visualization and clustering results, we derived the following insights:

1.  **Cluster Separability:** Styles such as 'Street' and 'Minimal' formed distinct, well-separated clusters in the latent space. This indicates that these styles possess unique visual features (e.g., color palette, complexity) that the VGG16 model can easily extract.
2.  **Feature Overlap:** The 'Casual' style showed significant overlap with other categories. This suggests that the definition of 'Casual' is visually broad and shares attributes with other styles, making it potentially difficult for a classifier to distinguish.
3.  **Future Work:**
    * The overlapping regions indicate hard-to-classify samples.
    * For the subsequent Supervised Learning (CNN) phase, we need to employ strong **Data Augmentation** and potentially use a deeper architecture (e.g., ResNet-50) to capture more subtle feature differences.